# Numerical Case Studies

This notebook turns the formulas from the lecture into executable algorithms. Every method is iterative, and every method has a different convergence story.

## Learning goals

- Estimate $\pi$ with two series and a Monte Carlo simulation.
- Measure absolute error against `math.pi`.
- Explain why Nilakantha converges faster than Gregory–Leibniz.
- Implement Newton's method with a tolerance and an iteration cap.
- Simulate coin tossing and distinguish proportions from counts.

In [ ]:
import math
import random

## 1. Gregory–Leibniz

$$
\pi = 4 \sum_{k=1}^{\infty} \frac{(-1)^{k+1}}{2k-1}
= 4\left(1 - \frac{1}{3} + \frac{1}{5} - \frac{1}{7} + \cdots\right).
$$

The terms decrease like $1/k$, so convergence is slow.

In [ ]:
def estimate_pi_gregory(terms):
    total = 0.0
    for k in range(1, terms + 1):
        total += (-1) ** (k + 1) / (2 * k - 1)
    return 4.0 * total

## 2. Nilakantha

$$
\pi = 3 + 4 \sum_{k=0}^{\infty} \frac{(-1)^k}{(2k+2)(2k+3)(2k+4)}.
$$

The terms decrease like $1/k^3$, so the same number of terms gives much more precision.

In [ ]:
def estimate_pi_nilakantha(terms):
    total = 3.0
    for k in range(terms):
        numerator = (-1) ** k
        denominator = (2 * k + 2) * (2 * k + 3) * (2 * k + 4)
        total += 4.0 * numerator / denominator
    return total

## 3. Comparing convergence

The table reports absolute error after `10`, `100`, `1_000`, and `10_000` terms.

In [ ]:
print(f"{'terms':>8} {'Gregory error':>16} {'Nilakantha error':>18}")
for terms in (10, 100, 1_000, 10_000):
    gregory = estimate_pi_gregory(terms)
    nilakantha = estimate_pi_nilakantha(terms)
    print(f"{terms:>8} {abs(gregory - math.pi):>16.3e} {abs(nilakantha - math.pi):>18.3e}")

Gregory–Leibniz needs roughly $1/(2\varepsilon)$ terms to reach an error $\varepsilon$. Nilakantha needs roughly $\varepsilon^{-1/3}$ terms. The difference is dramatic even at ten thousand terms.

## 4. Monte Carlo estimation of $\pi$

Sample points uniformly from the square $[-1, 1] \times [-1, 1]$. The square has area $4$; the inscribed circle has area $\pi$. Therefore

$$
\frac{A_c}{A_s} = \frac{\pi}{4}
\qquad\Longrightarrow\qquad
\pi \approx 4 \frac{N_c}{N}.
$$

The estimate is random. Its standard error is approximately $1.64/\sqrt{N}$.

In [ ]:
def estimate_pi_monte_carlo(samples, seed=0):
    random.seed(seed)
    inside = 0
    for _ in range(samples):
        x = random.uniform(-1.0, 1.0)
        y = random.uniform(-1.0, 1.0)
        if x * x + y * y <= 1.0:
            inside += 1
    return 4.0 * inside / samples

In [ ]:
print(f"{'samples':>9} {'mean error':>12} {'best error':>12}")
for samples in (1_000, 10_000, 100_000):
    errors = [abs(estimate_pi_monte_carlo(samples, seed=seed) - math.pi) for seed in range(5)]
    print(f"{samples:>9} {sum(errors) / len(errors):>12.6f} {min(errors):>12.6f}")

Halving the error requires roughly four times as many samples. Monte Carlo is easy to implement and generalises to high-dimensional integrals, but it converges slowly compared with a good deterministic method.

## 5. Euler's number

$$
e = \sum_{k=0}^{\infty} \frac{1}{k!}.
$$

The partial sums converge extremely quickly because $k!$ grows faster than any exponential.

In [ ]:
def estimate_e(terms):
    total = 0.0
    for k in range(terms):
        total += 1.0 / math.factorial(k)
    return total


for terms in (5, 10, 15, 20):
    estimate = estimate_e(terms)
    print(f"terms={terms:>2} estimate={estimate:.15f} error={abs(estimate - math.e):.3e}")

## 6. Newton's method

For $f(x) = x^2 - N$, Newton's update is

$$
x_{n+1} = x_n - \frac{x_n^2 - N}{2x_n}
= \frac{1}{2}\left(x_n + \frac{N}{x_n}\right).
$$

In [ ]:
def sqrt_newton(number, tolerance=1e-12, max_iterations=100):
    if number < 0:
        raise ValueError("number must be non-negative")
    if number == 0:
        return 0.0

    estimate = number / 2.0
    history = [estimate]
    for _ in range(max_iterations):
        next_estimate = (estimate + number / estimate) / 2.0
        history.append(next_estimate)
        if abs(next_estimate - estimate) < tolerance:
            return next_estimate, history
        estimate = next_estimate

    raise RuntimeError("Newton's method did not converge")


root, history = sqrt_newton(9)
for index, value in enumerate(history):
    print(f"n={index} x={value:.15f} error={abs(value - 3.0):.3e}")

The number of correct digits roughly doubles per iteration. The iteration cap prevents an infinite loop when the tolerance is unreachable, for example because of floating-point limits.

## 7. Coin tossing

A fair coin has $P(\text{heads}) = P(\text{tails}) = 1/2$. The law of large numbers describes the behaviour of the proportion, not the absolute count difference.

In [ ]:
def simulate_coin_tosses(tosses, seed=0):
    random.seed(seed)
    heads = 0
    tails = 0
    for _ in range(tosses):
        if random.random() < 0.5:
            tails += 1
        else:
            heads += 1
    return heads, tails


for tosses in (100, 1_000, 10_000, 100_000):
    heads, tails = simulate_coin_tosses(tosses)
    proportion = heads / tosses
    print(
        f"tosses={tosses:>7} heads={heads:>7} tails={tails:>7} "
        f"proportion={proportion:.5f} difference={heads - tails:>7}"
    )

The absolute difference usually grows with the number of tosses, while the proportion moves closer to $1/2$. This is why a large sample is described as precise in relative terms, not as a guarantee that counts will balance.

## Exercises

1. Estimate $\pi$ with Gregory–Leibniz using `100`, `1_000`, and `100_000` terms. Plot or print the error trend.
2. Repeat the Monte Carlo experiment with ten seeds and report the standard deviation of the estimates.
3. Implement the cosine series and compare it with `math.cos`.
4. Implement the arctangent series for $|x| < 1$ and use it to estimate $\pi$ with $x = 1$ cautiously.
5. Modify `sqrt_newton` to stop after a fixed number of iterations and return the history. Explain the trade-off.
6. Simulate a biased coin with probability `0.7` of heads and verify the long-run proportion.